# UC-11 — gold (Databricks)

Bouwt twee marts in `uwv_databricks.gold.*`:
1. `mart_uc11_klantreis_events` — UNION ALL over 7 silver-domeinen + row_number
2. `mart_uc11_klantreis_phases` — gaps-and-islands fase-reconstructie

Spark-SQL port van `dbt/models/marts/uc11_klantreis/mart_uc11_klantreis_*.sql`.
Identical aan de Fabric versie — alleen 3-part UC namen i.p.v. abfss paths.

In [ ]:
dbutils.widgets.text("catalog", "uwv_databricks")
catalog = dbutils.widgets.get("catalog")
spark.sql(f"USE CATALOG {catalog}")
print(f"Catalog: {catalog}")

In [ ]:
events_sql = f"""
CREATE OR REPLACE TABLE {catalog}.gold.mart_uc11_klantreis_events AS
WITH events AS (
    SELECT bsn, CAST(ingestion_source_ts AS TIMESTAMP) AS event_ts, event_date,
           'persoon' AS domein, 'persoon.aangemaakt' AS event_type,
           'Cliënt-record aangemaakt' AS event_label,
           CAST(NULL AS STRING) AS event_status, CAST(NULL AS STRING) AS regio_code,
           CAST(NULL AS DOUBLE) AS numeric_value, bsn AS source_ref_id
      FROM {catalog}.silver.persona
    UNION ALL
    SELECT bsn, CAST(aanvang_dienstverband AS TIMESTAMP) AS event_ts,
           aanvang_dienstverband AS event_date,
           'polisadm' AS domein, 'polisadm.ikv.start' AS event_type,
           CONCAT('Dienstverband begonnen bij ', COALESCE(werkgever_naam, '(onbekend)')) AS event_label,
           CAST(NULL AS STRING), CAST(NULL AS STRING),
           CAST(loon_bruto_jaar AS DOUBLE), ikv_id
      FROM {catalog}.silver.polisadm_ikv
     WHERE aanvang_dienstverband IS NOT NULL
    UNION ALL
    SELECT bsn, CAST(einde_dienstverband AS TIMESTAMP) AS event_ts,
           einde_dienstverband AS event_date,
           'polisadm' AS domein, 'polisadm.ikv.einde' AS event_type,
           CONCAT('Dienstverband beëindigd bij ', COALESCE(werkgever_naam, '(onbekend)')) AS event_label,
           CAST(NULL AS STRING), CAST(NULL AS STRING),
           CAST(NULL AS DOUBLE), ikv_id
      FROM {catalog}.silver.polisadm_ikv
     WHERE einde_dienstverband IS NOT NULL
    UNION ALL
    SELECT bsn, CAST(aanvraag_datum AS TIMESTAMP) AS event_ts,
           aanvraag_datum AS event_date,
           'ww' AS domein, CONCAT('ww.aanvraag.', LOWER(COALESCE(status, 'onbekend'))) AS event_type,
           CONCAT('WW-aanvraag (status: ', COALESCE(status, '?'),
                  ', reden: ', COALESCE(reden_einde_dienstverband, '?'), ')') AS event_label,
           status, CAST(NULL AS STRING), CAST(NULL AS DOUBLE), aanvraag_id
      FROM {catalog}.silver.ww_aanvraag
    UNION ALL
    SELECT bsn, CAST(eerste_ziektedag AS TIMESTAMP) AS event_ts,
           eerste_ziektedag AS event_date,
           'zw' AS domein, 'zw.melding' AS event_type,
           CONCAT('Ziekmelding (duur ~', CAST(COALESCE(duur_dagen, 0) AS STRING), ' dagen)') AS event_label,
           CAST(NULL AS STRING), CAST(NULL AS STRING),
           CAST(duur_dagen AS DOUBLE), melding_id
      FROM {catalog}.silver.zw_melding
    UNION ALL
    SELECT bsn, CAST(aanvraag_datum AS TIMESTAMP) AS event_ts,
           aanvraag_datum AS event_date,
           'wia' AS domein, CONCAT('wia.aanvraag.', LOWER(COALESCE(status, 'onbekend'))) AS event_type,
           CONCAT('WIA-aanvraag (', COALESCE(onderdeel, '?'),
                  ', status: ', COALESCE(status, '?'),
                  CASE WHEN arbeidsongeschikt_pct IS NOT NULL
                       THEN CONCAT(', AO ', CAST(arbeidsongeschikt_pct AS STRING), '%')
                       ELSE '' END, ')') AS event_label,
           status, regio_code, CAST(arbeidsongeschikt_pct AS DOUBLE), aanvraag_id
      FROM {catalog}.silver.wia_aanvraag
    UNION ALL
    SELECT bsn, CAST(ingangsdatum AS TIMESTAMP) AS event_ts,
           ingangsdatum AS event_date,
           'wajong' AS domein, 'wajong.dossier.geopend' AS event_type,
           CONCAT('Wajong-dossier (', COALESCE(regime, '?'),
                  ', arbeidsvermogen: ', COALESCE(arbeidsvermogen, '?'), ')') AS event_label,
           CAST(NULL AS STRING), CAST(NULL AS STRING),
           CAST(NULL AS DOUBLE), dossier_id
      FROM {catalog}.silver.wajong_dossier
     WHERE ingangsdatum IS NOT NULL
    UNION ALL
    SELECT bsn, CAST(contact_ts AS TIMESTAMP) AS event_ts,
           CAST(contact_ts AS DATE) AS event_date,
           'crm' AS domein,
           CONCAT('crm.contact.', LOWER(COALESCE(kanaal, 'onbekend'))) AS event_type,
           CONCAT('Klantcontact ', COALESCE(kanaal, '?'),
                  ' over ', COALESCE(onderwerp, '?')) AS event_label,
           CAST(NULL AS STRING), CAST(NULL AS STRING),
           CAST(duur_seconden AS DOUBLE), contact_id
      FROM {catalog}.silver.crm_contact
)
SELECT bsn,
       ROW_NUMBER() OVER (PARTITION BY bsn ORDER BY event_ts, domein, event_type) AS event_seq,
       event_ts, event_date, domein, event_type, event_label, event_status,
       regio_code, numeric_value, source_ref_id
  FROM events
"""
spark.sql(events_sql)
n = spark.table(f"{catalog}.gold.mart_uc11_klantreis_events").count()
print(f"gold.mart_uc11_klantreis_events: {n} rijen")

In [ ]:
phases_sql = f"""
CREATE OR REPLACE TABLE {catalog}.gold.mart_uc11_klantreis_phases AS
WITH with_fase AS (
    SELECT bsn, event_seq, event_ts, event_type, event_status,
        CASE
            WHEN event_type = 'polisadm.ikv.start'      THEN 'werknemer'
            WHEN event_type = 'polisadm.ikv.einde'      THEN 'tussen_dienstverband'
            WHEN event_type = 'zw.melding'              THEN 'ziek'
            WHEN event_type LIKE 'ww.aanvraag.%'
                 AND event_status IN ('INGEDIEND', 'IN_BEHANDELING') THEN 'ww_aanvraag'
            WHEN event_type = 'ww.aanvraag.toegekend'   THEN 'ww_uitkering'
            WHEN event_type = 'ww.aanvraag.afgewezen'   THEN 'tussen_dienstverband'
            WHEN event_type LIKE 'wia.aanvraag.%'
                 AND event_status IN ('INGEDIEND', 'IN_BEHANDELING') THEN 'wia_in_behandeling'
            WHEN event_type = 'wia.aanvraag.toegekend'  THEN 'wia_toegekend'
            WHEN event_type = 'wia.aanvraag.afgewezen'  THEN 'tussen_dienstverband'
            WHEN event_type = 'wajong.dossier.geopend'  THEN 'wajong_actief'
            ELSE NULL
        END AS fase
      FROM {catalog}.gold.mart_uc11_klantreis_events
),
fase_events AS (SELECT * FROM with_fase WHERE fase IS NOT NULL),
islands AS (
    SELECT bsn, event_seq, event_ts, fase, event_type AS trigger_event_type,
           LAG(fase) OVER (PARTITION BY bsn ORDER BY event_seq) AS prev_fase
      FROM fase_events
),
starts AS (
    SELECT bsn, event_seq, event_ts AS fase_start_ts, fase, trigger_event_type
      FROM islands WHERE prev_fase IS NULL OR prev_fase != fase
),
bounded AS (
    SELECT bsn,
           ROW_NUMBER() OVER (PARTITION BY bsn ORDER BY fase_start_ts) AS fase_volgnr,
           fase, trigger_event_type, fase_start_ts,
           LEAD(fase_start_ts) OVER (PARTITION BY bsn ORDER BY fase_start_ts) AS fase_eind_ts
      FROM starts
)
SELECT bsn, fase_volgnr, fase, trigger_event_type, fase_start_ts, fase_eind_ts,
       CASE WHEN fase_eind_ts IS NULL THEN NULL
            ELSE DATEDIFF(CAST(fase_eind_ts AS DATE), CAST(fase_start_ts AS DATE))
       END AS duur_dagen,
       CASE WHEN fase_eind_ts IS NULL THEN true ELSE false END AS is_lopend
  FROM bounded
"""
spark.sql(phases_sql)
n = spark.table(f"{catalog}.gold.mart_uc11_klantreis_phases").count()
print(f"gold.mart_uc11_klantreis_phases: {n} rijen")
print("Gold klaar.")